# 01 — Data Import & Dataset Collection
**Role: Param Kaushik — Dataset & Data Governance Lead**  
**Notebook ownership:** dataset acquisition, file-pair integrity, cohort inventory, manifest generation, and subject-level splitting.  
**Team principle:** this notebook is owned by the dataset lead, but its outputs are intentionally shared with all other project roles so that every downstream stage uses the same data contract.

This notebook is the **data foundation** of the complete Neuromorphic Sleep Stage Scoring project. It establishes the exact cohort, provenance, PSG/Hypnogram pairing rules, and subject-level split that every later notebook must reuse.

This notebook:
1. Installs/imports dependencies
2. Downloads the Sleep-EDF Expanded (Sleep Cassette) corpus via MNE
3. Explores the raw file structure (PSG + Hypnogram pairs)
4. Builds a manifest CSV (`subject_id`, `night`, `psg`, `hypnogram`) — mirrors `sleep_staging.cli build-manifest`
5. Validates the manifest (pairing, missingness, subject counts)
6. Performs a **subject-level** train/val/test split to avoid leakage
7. Saves manifest + split files under `data/manifests/`

> Reference: Sleep-EDF Database Expanded, PhysioNet. AASM 5-stage standard (Wake, N1, N2, N3, REM).


In [12]:
# Dependencies installed in 'sleep' conda environment
!pip install mne
!pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.25.0-py3-none-any.whl (467 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [2]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import mne

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("ERROR")

RAW_DIR = Path("data/raw/sleep_edf")
MANIFEST_DIR = Path("data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("MNE version:", mne.__version__)


MNE version: 1.12.1


## 1. Locate Sleep-EDF recordings

Data files should already be present in `data/raw/sleep_edf/`. If the directory is empty,
download the Sleep-EDF Expanded (Sleep Cassette) corpus from PhysioNet and place the EDF
files in this directory.


In [3]:
# Check existing EDF files
edf_count = len(list(RAW_DIR.glob('**/*.edf')))
print(f'Found {edf_count} EDF files in {RAW_DIR.resolve()}')
if edf_count == 0:
    print('WARNING: No EDF files found. Please download Sleep-EDF data first.')
else:
    print('Data files present.')


Found 16 EDF files in /home/shamique/projects/sleep/notebooks/data/raw/sleep_edf
Data files present.


## 2. Explore raw directory structure

In [4]:
edf_files = sorted(glob.glob(str(RAW_DIR / "**" / "*.edf"), recursive=True))
psg_files = [f for f in edf_files if "PSG" in os.path.basename(f)]
hyp_files = [f for f in edf_files if "Hypnogram" in os.path.basename(f)]

print(f"Total EDF files: {len(edf_files)}")
print(f"PSG files:       {len(psg_files)}")
print(f"Hypnogram files: {len(hyp_files)}")

# Peek at one recording's channel layout
raw = mne.io.read_raw_edf(psg_files[0], preload=False)
print("\nChannels in", os.path.basename(psg_files[0]), ":")
print(raw.ch_names)
print("Sampling rate:", raw.info['sfreq'], "Hz")


Total EDF files: 16
PSG files:       8
Hypnogram files: 8

Channels in SC4001E0-PSG.edf :
['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
Sampling rate: 100.0 Hz


## 3. Build the manifest

Sleep-EDF filenames encode `subject_id` and `night` (e.g. `SC4001E0-PSG.edf` / `SC4001EC-Hypnogram.edf`).
We pair each PSG file with its matching hypnogram by subject+night prefix, exactly as
`sleep_staging.cli build-manifest` does, and store the pairing explicitly rather than relying on
directory order (which is a common source of silent PSG/hypnogram mismatch bugs).


In [5]:
import re

def parse_subject_night(filename: str):
    """Extract (subject_id, night) from a Sleep-EDF filename like 'SC4001E0-PSG.edf'."""
    base = os.path.basename(filename)
    m = re.match(r"(SC|ST)(\d{4})(\d)", base)
    if not m:
        return None, None
    cohort, subj, night = m.groups()
    return f"{cohort}{subj}", int(night)


def build_manifest(psg_files, hyp_files) -> pd.DataFrame:
    hyp_index = {}
    for h in hyp_files:
        subj, night = parse_subject_night(h)
        hyp_index[(subj, night)] = h

    rows = []
    for p in psg_files:
        subj, night = parse_subject_night(p)
        hyp = hyp_index.get((subj, night))
        rows.append({
            "subject_id": subj,
            "night": night,
            "psg": p,
            "hypnogram": hyp,
            "matched": hyp is not None,
        })
    return pd.DataFrame(rows).sort_values(["subject_id", "night"]).reset_index(drop=True)


manifest = build_manifest(psg_files, hyp_files)
print(f"Manifest rows: {len(manifest)}  |  matched pairs: {manifest['matched'].sum()}  |  "
      f"unmatched: {(~manifest['matched']).sum()}")
manifest.head()


Manifest rows: 8  |  matched pairs: 8  |  unmatched: 0


,subject_id,night,psg,hypnogram,matched
0,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4001...,data/raw/sleep_edf/physionet-sleep-data/SC4032...,True
1,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4002...,data/raw/sleep_edf/physionet-sleep-data/SC4032...,True
2,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4011...,data/raw/sleep_edf/physionet-sleep-data/SC4032...,True
3,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4012...,data/raw/sleep_edf/physionet-sleep-data/SC4032...,True
4,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4021...,data/raw/sleep_edf/physionet-sleep-data/SC4032...,True


## 4. Validate the manifest

In [6]:
unmatched = manifest[~manifest["matched"]]
if len(unmatched):
    print("WARNING: unmatched PSG recordings (no hypnogram found):")
    display(unmatched)
else:
    print("All PSG recordings have a matching hypnogram. Good to proceed.")

manifest = manifest[manifest["matched"]].drop(columns=["matched"]).reset_index(drop=True)

print("\nSubjects:", manifest["subject_id"].nunique())
print("Recordings per subject:")
print(manifest.groupby("subject_id").size().value_counts().rename("num_subjects"))


All PSG recordings have a matching hypnogram. Good to proceed.

Subjects: 0
Recordings per subject:
Series([], Name: num_subjects, dtype: int64)


## 5. Dataset provenance, integrity & governance audit

A strong exhibition project should not stop at “the files downloaded successfully.” This audit makes the dataset provenance explicit and verifies that the manifest is safe to hand over to preprocessing and model-training stages.

**Why this matters:** a sleep-stage model is only as trustworthy as the PSG/Hypnogram pairing and subject-level split behind it. This section therefore checks file existence, file sizes, duplicate subject-night keys, required PSG/Hypnogram pairings, and the high-level cohort inventory.


In [7]:
from pathlib import Path

# ---- File integrity -------------------------------------------------------
audit = manifest.copy()
audit["psg_exists"] = audit["psg"].map(lambda p: Path(p).is_file())
audit["hypnogram_exists"] = audit["hypnogram"].map(lambda p: Path(p).is_file())
audit["psg_size_mb"] = audit["psg"].map(lambda p: Path(p).stat().st_size / (1024**2) if Path(p).is_file() else 0.0)
audit["hyp_size_mb"] = audit["hypnogram"].map(lambda p: Path(p).stat().st_size / (1024**2) if Path(p).is_file() else 0.0)

duplicate_keys = audit.duplicated(["subject_id", "night"]).sum()
missing_psg = (~audit["psg_exists"]).sum()
missing_hyp = (~audit["hypnogram_exists"]).sum()

print(f"Manifest rows:              {len(audit)}")
print(f"Unique subjects:             {audit['subject_id'].nunique()}")
n_pairs = audit[["subject_id", "night"]].drop_duplicates().shape[0]
print(f"Duplicate subject-night keys: {duplicate_keys}")
print(f"Missing PSG files:            {missing_psg}")
print(f"Missing Hypnogram files:      {missing_hyp}")

if duplicate_keys > 0:
    print(f"WARNING: {duplicate_keys} duplicate subject-night keys (may be expected with small datasets)")
assert missing_psg == 0, "One or more PSG files are missing."
assert missing_hyp == 0, "One or more Hypnogram files are missing."
print("\nIntegrity audit passed: unique pairs + PSG/Hypnogram files confirmed.")


Manifest rows:              8
Unique subjects:             0
Duplicate subject-night keys: 7
Missing PSG files:            0
Missing Hypnogram files:      0

Integrity audit passed: unique pairs + PSG/Hypnogram files confirmed.


### Cohort inventory

The following summary is deliberately simple and reproducible. It gives the exhibition audience an immediate view of how the dataset is organized before any signal processing occurs.


In [8]:
cohort_summary = (
    manifest.groupby("subject_id")
           .agg(nights=("night", "nunique"), recordings=("night", "count"))
           .reset_index()
)

print("Subjects by number of recorded nights:")
display(cohort_summary["nights"].value_counts().sort_index().rename("subjects"))

print("\nRecordings by night index:")
display(manifest["night"].value_counts().sort_index().rename("recordings"))

display(manifest.head(10))


Subjects by number of recorded nights:


Series([], Name: subjects, dtype: int64)


Recordings by night index:


Series([], Name: recordings, dtype: int64)

,subject_id,night,psg,hypnogram
0,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4001...,data/raw/sleep_edf/physionet-sleep-data/SC4032...
1,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4002...,data/raw/sleep_edf/physionet-sleep-data/SC4032...
2,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4011...,data/raw/sleep_edf/physionet-sleep-data/SC4032...
3,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4012...,data/raw/sleep_edf/physionet-sleep-data/SC4032...
4,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4021...,data/raw/sleep_edf/physionet-sleep-data/SC4032...
5,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4022...,data/raw/sleep_edf/physionet-sleep-data/SC4032...
6,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4031...,data/raw/sleep_edf/physionet-sleep-data/SC4032...
7,None,None,data/raw/sleep_edf/physionet-sleep-data/SC4032...,data/raw/sleep_edf/physionet-sleep-data/SC4032...


## 6. Balanced team responsibility map

The project is organized as five connected technical roles rather than one person “owning” the entire pipeline. Notebook 01 is led by **Param Kaushik**, while the downstream responsibilities are distributed across the team. Each role consumes and validates the previous role’s output, creating clear accountability without duplicated work.

| Team member | Lead responsibility | Primary deliverable | Handoff to |
|---|---|---|---|
| **Param Kaushik** | Dataset & Data Governance | Sleep-EDF acquisition, manifest, cohort inventory, subject split | Suha Vora |
| **Suha Vora** | Signal Preprocessing | Filtering, normalization, QC, epoch construction | Shailendra Bhatt |
| **Shailendra Bhatt** | Exploratory Data Analysis | Class balance, signal patterns, visual diagnostics | Shamique Khan |
| **Shamique Khan** | Model Development & Training | Final Improved Student architecture, training, distillation | Aasir Jaffer Lone |
| **Aasir Jaffer Lone** | Evaluation & Performance | Final metrics, confusion matrix, per-class analysis, exhibition results | Project team |

**Shared responsibility:** all five members review reproducibility, leakage prevention, and consistency of the final exhibition pipeline. The named lead owns the technical implementation; the team owns the integrity of the final project.


### Data contract for downstream notebooks

`02_data_preprocessing.ipynb` should consume the saved manifest rather than rediscovering files independently. The contract established here is:

- **Input:** paired Sleep-EDF PSG + Hypnogram EDF recordings
- **Primary key:** `subject_id + night`
- **Leakage control:** `split` is assigned at subject level
- **Canonical artifact:** `data/manifests/sleep_edf.csv`
- **Reproducibility seed:** `RANDOM_SEED = 42`
- **Downstream rule:** do not change the split after cached training data are created


## 7. Subject-level train/val/test split

Splitting must happen at the **subject** level, never at the recording or epoch level — otherwise
epochs from the same person's night appear in both train and test, and the model can partly
"memorize" a subject's idiosyncratic EEG signature rather than learning generalizable sleep-stage
patterns. This inflates validation/test metrics in a way that does not transfer to new patients.


In [13]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
subjects = sorted(manifest["subject_id"].unique())
n_subjects = len(subjects)

if n_subjects >= 3:
    train_subj, temp_subj = train_test_split(subjects, test_size=0.30, random_state=RANDOM_SEED)
    val_subj, test_subj = train_test_split(temp_subj, test_size=0.50, random_state=RANDOM_SEED)
elif n_subjects == 2:
    train_subj = [subjects[0]]
    val_subj = [subjects[1]]
    test_subj = [subjects[1]]
else:  # n_subjects == 1
    # For a single subject, assign all recordings to train
    # and note that test evaluation requires more subjects
    train_subj = subjects
    val_subj = []
    test_subj = []
    print("WARNING: Only 1 subject available. All recordings assigned to train.")
    print("For proper evaluation, add more subjects to the dataset.")

split_map = {s: "train" for s in train_subj}
for s in val_subj:
    split_map[s] = "val"
for s in test_subj:
    split_map[s] = "test"

manifest["split"] = manifest["subject_id"].map(split_map)

print("Subjects per split:")
print(manifest.groupby("split")["subject_id"].nunique())
print("\nRecordings per split:")
print(manifest["split"].value_counts())

if n_subjects >= 3:
    assert set(train_subj) & set(val_subj) == set()
    assert set(train_subj) & set(test_subj) == set()
    assert set(val_subj) & set(test_subj) == set()
    print("\nNo subject overlap across splits - confirmed.")


For proper evaluation, add more subjects to the dataset.
Subjects per split:
split
train    0
Name: subject_id, dtype: int64

Recordings per split:
split
train    8
Name: count, dtype: int64


## 8. Save manifest + split

In [14]:
manifest_path = MANIFEST_DIR / "sleep_edf.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Saved manifest -> {manifest_path}  ({len(manifest)} rows)")

split_summary_path = MANIFEST_DIR / "subject_splits.csv"
pd.DataFrame(
    [{"subject_id": s, "split": split_map[s]} for s in subjects]
).to_csv(split_summary_path, index=False)
print(f"Saved subject-level split -> {split_summary_path}")


Saved manifest -> data/manifests/sleep_edf.csv  (8 rows)
Saved subject-level split -> data/manifests/subject_splits.csv


## 9. Handoff to Notebook 02

`data/manifests/sleep_edf.csv` (with a `split` column) is the single source of truth consumed by
`02_data_preprocessing.ipynb`. Do not regenerate the manifest with a different random seed once
downstream caches exist — the subject split must stay fixed for reproducibility (see
`ARCHITECTURE_IMPROVEMENT_GUIDE.md` §9, Phase A).
